
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Appendix: Agent Bricks CLI
#### Duration: 30 minutes (optional)

## Overview

The **Agent Bricks CLI** (`databricks-agentbricks`) is a command-line tool for building and deploying custom agents in code. It scaffolds a project from a framework template, runs it locally, and deploys it to the Databricks agent runtime. Along the way, it handles model access, memory, sessions, tools, and tracing so you can focus on your agent's logic.

This appendix introduces the core concepts and commands so you know what the CLI does and when you might reach for it.

## Learning Objectives
By the end of this appendix you will be able to:
- Describe what the Agent Bricks CLI manages for you and what you still write
- Explain the `init`, `dev`, and `deploy` lifecycle
- Explain the role of `agent.toml` as the declarative source of truth for agent resources
- Compare the CLI with deploying an agent as a Databricks App with Declarative Automation Bundles (DABs)

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">The Agent Bricks CLI is in <strong>Beta</strong>, so its commands and behavior may change. Treat this appendix as reference material and check the docs for the latest details.</p>
</div>

## A. What the CLI Does for You

When you build an agent with the Agent Bricks CLI, the work splits into two parts. You write the application logic, and the CLI wires that logic up to Databricks-managed resources.

<div style="display: flex; gap: 18px; flex-wrap: wrap; justify-content: center; align-items: stretch; margin: 20px 0; font-family: sans-serif;">
  <div style="flex: 1 1 260px; max-width: 320px; box-sizing: border-box; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>
    <div style="display: inline-block; background: #FF5F46; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">YOU WRITE</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Application logic</div>
    <ul style="font-size: 14px; color: #5A6F77; margin: 8px 0 0 0; padding-left: 18px; line-height: 1.7;">
      <li>Model choice</li>
      <li>Prompts</li>
      <li>Custom Python tools</li>
    </ul>
  </div>
  <div style="display: flex; align-items: center; font-size: 28px; color: #5A6F77;">+</div>
  <div style="flex: 1 1 260px; max-width: 320px; box-sizing: border-box; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #2272B4;"></div>
    <div style="display: inline-block; background: #2272B4; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">CLI MANAGES</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Databricks resources</div>
    <ul style="font-size: 14px; color: #5A6F77; margin: 8px 0 0 0; padding-left: 18px; line-height: 1.7;">
      <li>Model access</li>
      <li>Managed memory and sessions</li>
      <li>Managed tools (MCP, UC functions, sandbox)</li>
      <li>MLflow tracing</li>
      <li>Deployment and permissions</li>
    </ul>
  </div>
</div>

| Capability | What the CLI handles |
|---|---|
| **Model access** | Provisions access to a Databricks-served model, so you don't manage credentials or endpoints. |
| **Managed memory** | Long-term memories the agent can write and search, partitioned by actor. Persists facts and preferences across sessions. |
| **Managed sessions** | Conversation transcripts held in managed session stores, partitioned by actor. Sessions can be forked into independent copies. |
| **Tools** | Databricks-managed tools declared in `agent.toml`: a downscoped Unity Catalog sandbox, a Databricks-managed MCP service, or a Unity Catalog function. |
| **Tracing** | MLflow tracing is on by default and sends each run's traces to a per-project MLflow experiment. |
| **Deployment** | Deploys to the Databricks agent runtime and grants the agent's service principal access to its bound stores. |

## B. The Three-Command Lifecycle

Three commands take an agent from an empty directory to production.

<div style="display: flex; gap: 10px; flex-wrap: wrap; justify-content: center; align-items: stretch; margin: 20px 0; font-family: sans-serif;">
  <div style="flex: 1 1 200px; max-width: 240px; box-sizing: border-box; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #2272B4;"></div>
    <div style="display: inline-block; background: #2272B4; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">1. SCAFFOLD</div>
    <div style="font-size: 16px; font-weight: 700; color: #0b2026; font-family: monospace;">agentbricks init</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Creates the project from a framework template and writes <code>agent.toml</code>.</div>
  </div>
  <div style="display: flex; align-items: center; font-size: 28px; color: #5A6F77;">&rarr;</div>
  <div style="flex: 1 1 200px; max-width: 240px; box-sizing: border-box; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #00A972;"></div>
    <div style="display: inline-block; background: #00A972; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">2. RUN LOCALLY</div>
    <div style="font-size: 16px; font-weight: 700; color: #0b2026; font-family: monospace;">agentbricks dev</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Runs the agent on port 8000 with the same runtime and environment as Databricks.</div>
  </div>
  <div style="display: flex; align-items: center; font-size: 28px; color: #5A6F77;">&rarr;</div>
  <div style="flex: 1 1 200px; max-width: 240px; box-sizing: border-box; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>
    <div style="display: inline-block; background: #FF5F46; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">3. DEPLOY</div>
    <div style="font-size: 16px; font-weight: 700; color: #0b2026; font-family: monospace;">agentbricks deploy</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Provisions the resources in <code>agent.toml</code> and rolls out to the agent runtime.</div>
  </div>
</div>

### What it looks like

**Prerequisites:** the Databricks CLI on your path, and Python 3.10 or above with pip.

The CLI ships two framework templates: `langgraph` for LangGraph and `openai` for the OpenAI Agents SDK. The examples below use LangGraph, matching the docs.

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="20" height="17" viewBox="0 0 20 17" fill="none" style="display:inline-block;vertical-align:middle;"><rect x="1.5" y="1" width="17" height="15" rx="2.5" stroke="#5A6F77" stroke-width="2"/><path d="M5.5 6L8.5 8.5L5.5 11" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/><line x1="10.5" y1="11" x2="14.5" y2="11" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Terminal</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># Install the Agent Bricks CLI
pip install databricks-agentbricks
<span></span>
# Authenticate with OAuth and set the profile as the CLI's default
databricks auth login --host https://&amp;lt;your-workspace-url&amp;gt; --profile &amp;lt;profile&amp;gt;
agentbricks login --profile &amp;lt;profile&amp;gt;
<span></span>
# 1. Scaffold a project (templates: langgraph or openai)
agentbricks init --framework langgraph my-agent
cd my-agent
<span></span>
# 2. Run locally at http://localhost:8000
agentbricks dev
<span></span>
# 3. Deploy to the Databricks agent runtime
agentbricks deploy my-agent</div>
</div>

When the deployment finishes, the CLI returns the URL of the live agent. The deployed agent is named `agent-bricks-&lt;name&gt;`.

For the step-by-step walkthrough, see: Agent Bricks quickstart (<a href="https://docs.databricks.com/aws/en/agents/quickstart" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/quickstart" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/quickstart" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

## C. agent.toml: The Source of Truth

`agent.toml` declares every Databricks-managed resource your agent depends on. CLI commands write to it, and `agentbricks deploy` reads it to provision and wire everything up. The file, not hand-written setup code, is what deploys.

<div style="display: flex; gap: 10px; flex-wrap: wrap; justify-content: center; align-items: center; margin: 20px 0; font-family: sans-serif;">
  <div style="flex: 1 1 220px; max-width: 260px; box-sizing: border-box; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 14px 16px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="display: inline-block; background: #5A6F77; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">WRITE</div>
    <div style="font-size: 13px; color: #0b2026; font-family: monospace; line-height: 1.9;">agentbricks init<br>agentbricks tools add<br>agentbricks memory bind<br>agentbricks sessions bind<br>agentbricks tracing bind</div>
  </div>
  <div style="font-size: 28px; color: #5A6F77;">&rarr;</div>
  <div style="flex: 0 1 180px; box-sizing: border-box; background: #1B5162; color: #fff; border-radius: 14px; padding: 22px 16px; text-align: center; box-shadow: 0 2px 8px rgba(27,49,57,.15);">
    <div style="font-size: 18px; font-weight: 700; font-family: monospace;">agent.toml</div>
    <div style="font-size: 13px; margin-top: 6px; opacity: 0.85;">declares resources</div>
  </div>
  <div style="font-size: 28px; color: #5A6F77;">&rarr;</div>
  <div style="flex: 1 1 220px; max-width: 260px; box-sizing: border-box; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 14px 16px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="display: inline-block; background: #FF5F46; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">DEPLOY</div>
    <div style="font-size: 13px; color: #0b2026; font-family: monospace; margin-bottom: 6px;">agentbricks deploy</div>
    <ul style="font-size: 13px; color: #5A6F77; margin: 0; padding-left: 18px; line-height: 1.7;">
      <li>Creates stores</li>
      <li>Binds tools</li>
      <li>Grants access</li>
      <li>Rolls out agent</li>
    </ul>
  </div>
</div>

### A generated agent.toml

This is the file that `agentbricks init` generates for a LangGraph agent named `my-agent`:

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">TOML</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">schema_version = 1
<span></span>
[agent]
framework = "langgraph"
server = "agentbricks"
<span></span>
[memory_store]
name = "my-agent-memory"
<span></span>
[session_store]
name = "my-agent-session"
<span></span>
[tracing]
experiment_name = "/Shared/agentbricks_traces/my-agent"</div>
</div>

### Updating it with CLI commands

You can add tools and bind stores at any point, not only at `init`. You can also edit `agent.toml` directly.

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="20" height="17" viewBox="0 0 20 17" fill="none" style="display:inline-block;vertical-align:middle;"><rect x="1.5" y="1" width="17" height="15" rx="2.5" stroke="#5A6F77" stroke-width="2"/><path d="M5.5 6L8.5 8.5L5.5 11" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/><line x1="10.5" y1="11" x2="14.5" y2="11" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Terminal</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># Bind managed session and memory stores (created if they don't exist)
agentbricks sessions bind my-agent-session
agentbricks memory bind my-agent-memory
<span></span>
# Add a built-in MCP service as a tool
agentbricks tools list --kind mcp
agentbricks tools add mcp system.ai.web_search
<span></span>
# View traces from local and deployed runs
agentbricks tracing list</div>
</div>

By default, a tool runs with the permissions of the user who sent the request. Add `--auth app` to run it as the agent's service principal instead.

## D. Bringing an Existing Agent

If you already built an agent with LangGraph or the OpenAI Agents SDK, the `--existing` flag helps you move it to the Agent Bricks CLI. The CLI doesn't rewrite your code. Instead, it prepares migration instructions for a coding agent, such as Claude Code or Codex, to follow.

| Step | Command or action | What happens |
|---|---|---|
| **1. Prepare** | `agentbricks init --framework langgraph --existing .` | Writes an `agent-bricks-migrate/` directory with instructions, a prompt, and a reference project. Doesn't change your code or create workspace resources. |
| **2. Convert** | Paste the prompt into your coding agent | The coding agent converts the project to use `agent.toml` and a `DurableAgentServer` entrypoint. |
| **3. Check** | `agentbricks doctor .` | Inspects the project files without running code or contacting Databricks. A failed report means the conversion isn't finished. |
| **4. Ship** | `agentbricks dev`, then `agentbricks deploy` | Delete `agent-bricks-migrate/` and its skills first, then run and deploy as usual. |

Switching to a managed session store doesn't move existing conversation history, so decide how to handle earlier conversations before you migrate.

## E. Comparing the CLI with Apps and DABs

A common way to deploy an agent is as a Databricks App with Declarative Automation Bundles (DABs). The Agent Bricks CLI is a code-first alternative that declares the agent's resources in `agent.toml` and provisions them on deploy.

| | Apps + DABs | Agent Bricks CLI |
|---|---|---|
| **Resource wiring** | Declared in `databricks.yml` | Declared in `agent.toml` |
| **Deploy** | `databricks bundle deploy`, then `databricks apps deploy` | `agentbricks deploy` |
| **MCP tools** | Connected in agent code with `McpServer` | `agentbricks tools add mcp &lt;service&gt;`, connected at runtime |
| **Tracing** | Added with autologging and `@mlflow.trace` | On by default, sent to a per-project MLflow experiment |

Because traces from a CLI-deployed agent land in a standard MLflow experiment, the same tracing, scorer, and online evaluation techniques apply.

### Learn more

- Agent Bricks CLI (<a href="https://docs.databricks.com/aws/en/agents/custom-agents/agent-bricks-cli" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/custom-agents/agent-bricks-cli" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/custom-agents/agent-bricks-cli" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)
- Agent memory (<a href="https://docs.databricks.com/aws/en/agents/agent-memory/managed-memory" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/agent-memory/managed-memory" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/agent-memory/managed-memory" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)
- Agent sessions (<a href="https://docs.databricks.com/aws/en/agents/agent-memory/managed-sessions" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/agent-memory/managed-sessions" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/agent-memory/managed-sessions" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)
- MCPs (<a href="https://docs.databricks.com/aws/en/agents/mcp-tools/" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/mcp-tools/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/mcp-tools/" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)
- Tracing overview (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/overview" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/overview" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/overview" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)
- Full command reference: <a href="https://github.com/databricks/databricks-ai-bridge/blob/main/integrations/agentbricks/README.md" target="_blank" style="color: #1976d2; text-decoration: underline;">Agent Bricks CLI README on GitHub</a>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="color: #1976d2; text-decoration: underline;">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="color: #1976d2; text-decoration: underline;">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="color: #1976d2; text-decoration: underline;">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="color: #1976d2; text-decoration: underline;">Support</a>
